# Exploratory data analysis

Profile the raw student survey, check missing values and anomalies, and visualize feature distributions and their relationships with mental-health risk. This notebook reports anomalies without removing rows.


In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "student_mental_health.csv"
student_df = pd.read_csv(data_path)


In [ ]:
print("Aper?u des premi?res lignes :")
display(student_df.head())

print(f"\nDimensions du jeu de donn?es : {student_df.shape[0]} lignes et {student_df.shape[1]} colonnes\n")
student_df.info()


In [ ]:
missing_counts = student_df.isna().sum()
missing_percent = (missing_counts / len(student_df) * 100).round(2)
missing_df = pd.DataFrame(
    {"Valeurs manquantes": missing_counts, "Pourcentage (%)": missing_percent}
)

print("Colonnes avec des valeurs manquantes :")
display(
    missing_df[missing_df["Valeurs manquantes"] > 0]
    .sort_values("Valeurs manquantes", ascending=False)
)
print(f"\nNombre de lignes dupliqu?es : {student_df.duplicated().sum()}")


In [ ]:
print("Statistiques des variables num?riques :")
display(student_df.describe(include="number").T)


In [ ]:
anomalies = {
    "?ge (<15 ou >40)": ((student_df["Age"] < 15) | (student_df["Age"] > 40)).sum(),
    "Sommeil (<0 ou >24 h)": ((student_df["Sleep_Hours"] < 0) | (student_df["Sleep_Hours"] > 24)).sum(),
    "GPA (<0 ou >4)": ((student_df["GPA"] < 0) | (student_df["GPA"] > 4)).sum(),
    "Pr?sence (<0 ou >100 %)": (
        (student_df["Attendance_Rate"] < 0) | (student_df["Attendance_Rate"] > 100)
    ).sum(),
    "R?seaux sociaux > temps d'?cran": (
        student_df["Social_Media_Hours"] > student_df["Screen_Time_Hours"]
    ).sum(),
}

for anomaly, count in anomalies.items():
    print(f"{anomaly}: {count}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_columns = student_df.select_dtypes(include="number").columns

for column in numeric_columns:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=student_df[column], color="skyblue")
    plt.title(f"Distribution de {column}")
    plt.xlabel("")
    plt.tight_layout()
    plt.show()


In [ ]:
outliers_summary = []

for column in numeric_columns:
    q1 = student_df[column].quantile(0.25)
    q3 = student_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = student_df.loc[
        (student_df[column] < lower_bound) | (student_df[column] > upper_bound),
        column,
    ]

    outliers_summary.append(
        {
            "Variable": column,
            "Borne inf?rieure": round(lower_bound, 2),
            "Borne sup?rieure": round(upper_bound, 2),
            "Nombre d'outliers": len(outliers),
            "Valeurs extr?mes": sorted(outliers.tolist()),
        }
    )

display(pd.DataFrame(outliers_summary))


In [ ]:
numeric_stats = student_df[numeric_columns].describe().T
numeric_stats["median"] = student_df[numeric_columns].median()
numeric_stats = numeric_stats[
    ["count", "mean", "std", "min", "25%", "median", "75%", "max"]
]

print("Statistiques des variables num?riques :")
display(numeric_stats)

categorical_columns = student_df.select_dtypes(exclude="number").columns
print("Statistiques des variables cat?gorielles :")
display(student_df[categorical_columns].describe().T)


In [ ]:
print(student_df["Mental_Health_Risk"].value_counts())
sns.countplot(data=student_df, x="Mental_Health_Risk")
plt.show()


In [ ]:
plt.figure(figsize=(14, 10))
for index, column in enumerate(numeric_columns, 1):
    plt.subplot(4, 3, index)
    sns.histplot(student_df[column], kde=True)
    plt.title(column)

plt.tight_layout()
plt.show()


In [ ]:
mean_by_risk = (
    student_df.groupby("Mental_Health_Risk")[numeric_columns]
    .mean()
    .loc[["Low", "Medium", "High"]]
    .reset_index()
)
means_long = mean_by_risk.melt(
    id_vars="Mental_Health_Risk",
    var_name="feature",
    value_name="mean_value",
)

plt.figure(figsize=(20, 10))
sns.barplot(
    data=means_long,
    x="feature",
    y="mean_value",
    hue="Mental_Health_Risk",
    palette="Set2",
)
plt.title("Valeur moyenne des facteurs par niveau de risque", fontweight="bold")
plt.show()


In [ ]:
correlation_matrix = student_df[numeric_columns].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
plt.title("Corr?lation entre variables num?riques")
plt.tight_layout()
plt.show()


In [ ]:
risk_order = {"Low": 0, "Medium": 1, "High": 2}
student_df["mental_health_risk_ordinal"] = student_df["Mental_Health_Risk"].map(risk_order)
print(
    student_df.select_dtypes(include="number")
    .corr()["mental_health_risk_ordinal"]
)
